# 04 — PDP / ICE Plot
Effet de `friends_followers_ratio` sur la probabilité prédite d'être un bot.
- **ICE** (Individual Conditional Expectation) : une courbe par compte, colorée par label réel
- **PDP** (Partial Dependence Plot) : moyenne de toutes les courbes ICE

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')
from data import load_dataset_split, FEATURES

In [ ]:
X_train, X_test, y_train, y_test = load_dataset_split()
model = joblib.load('../models/xgboost.joblib')
print('Dataset test :', X_test.shape)

## Calcul des courbes ICE
Pour chaque compte, on fait varier `friends_followers_ratio` sur une grille de valeurs
et on observe comment la probabilité prédite évolue — toutes les autres features restent fixes.

In [ ]:
feat = 'friends_followers_ratio'
feat_idx = FEATURES.index(feat)

# Grille de valeurs pour la feature (percentile 5 → 95)
grid = np.linspace(
    X_test[feat].quantile(0.05),
    X_test[feat].quantile(0.95),
    60
)
print(f'Grille : {grid.min():.2f} → {grid.max():.2f} ({len(grid)} points)')

In [ ]:
# Échantillon de 300 comptes (150 bots + 150 humains)
idx_bots   = y_test[y_test == 1].sample(150, random_state=42).index
idx_humans = y_test[y_test == 0].sample(150, random_state=42).index
sample_idx = idx_bots.append(idx_humans)

X_sample = X_test.loc[sample_idx].copy()
y_sample = y_test.loc[sample_idx]
print(f'Échantillon : {len(X_sample)} comptes ({y_sample.sum()} bots, {(y_sample==0).sum()} humains)')

In [ ]:
# Calcul des courbes ICE
ice_curves = np.zeros((len(X_sample), len(grid)))

for j, val in enumerate(grid):
    X_tmp = X_sample.copy()
    X_tmp[feat] = val
    ice_curves[:, j] = model.predict_proba(X_tmp)[:, 1]

print('ICE calculées :', ice_curves.shape)

## Visualisation — ICE colorées par label réel

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

labels = y_sample.values

# ICE humains
for i, label in enumerate(labels):
    color = '#EF4444' if label == 1 else '#06B6D4'
    alpha = 0.06
    ax.plot(grid, ice_curves[i], color=color, alpha=alpha, linewidth=0.8)

# PDP = moyenne de toutes les courbes ICE
pdp = ice_curves.mean(axis=0)
ax.plot(grid, pdp, color='#F59E0B', linewidth=3.5, label='PDP — effet moyen', zorder=5)

# PDP par classe
pdp_bot   = ice_curves[labels == 1].mean(axis=0)
pdp_human = ice_curves[labels == 0].mean(axis=0)
ax.plot(grid, pdp_bot,   color='#EF4444', linewidth=2, linestyle='--', label='PDP bots', zorder=4)
ax.plot(grid, pdp_human, color='#06B6D4', linewidth=2, linestyle='--', label='PDP humains', zorder=4)

# Seuil suspect
ax.axvline(x=5, color='white', linestyle=':', linewidth=1.5, label='Seuil suspect (ratio = 5)')

# Légende proxy pour les ICE
from matplotlib.lines import Line2D
handles, labels_leg = ax.get_legend_handles_labels()
handles += [
    Line2D([0], [0], color='#EF4444', alpha=0.4, linewidth=2, label='ICE — bots'),
    Line2D([0], [0], color='#06B6D4', alpha=0.4, linewidth=2, label='ICE — humains'),
]
ax.legend(handles=handles, fontsize=10)

ax.set_xlabel('friends_followers_ratio (abonnements / followers)', fontsize=12)
ax.set_ylabel('Probabilité prédite — bot', fontsize=12)
ax.set_title('PDP / ICE — effect de friends_followers_ratio sur la prédiction', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/pdp_ice_friends_followers.png', dpi=150)
plt.show()
print('Sauvegardé : plots/pdp_ice_friends_followers.png')

## Lecture du graphique
- **Lignes rouges** : comptes bots — leur probabilité prédite augmente fortement avec le ratio
- **Lignes bleues** : comptes humains — leur probabilité reste basse même avec un ratio élevé
- **Ligne orange** : PDP global — l'effet moyen toutes classes confondues
- **Ligne pointillée blanche** : seuil 5 au-delà duquel le modèle classe quasi systématiquement en bot